# Project 4: NLP & Sentiment Analysis

This notebook follows the complete pipeline: preprocessing, TF-IDF vectorization, model training, evaluation and prediction.

In [1]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path("data/reviews.csv")
df = pd.read_csv(DATA_PATH)
df.head()

,review_id,review,sentiment
0,P117,I would recommend this laptop. It is easy to u...,positive
1,P001,This backpack is not bad at all. It feels exce...,positive
2,N079,Poor value for money. The tablet feels bad and...,negative
3,P152,Good value for money. The watch feels impressi...,positive
4,N107,"After one week, the camera started giving prob...",negative


## Step 1: Text Preprocessing

The preprocessing pipeline cleans raw reviews, tokenizes them, removes stop-words while preserving negations, and applies POS-guided lemmatization.

In [2]:
import sys
sys.path.append("src")
from preprocessing import download_nltk_resources, preprocess_text

download_nltk_resources()
df["clean_review"] = df["review"].apply(preprocess_text)
df[["review", "clean_review", "sentiment"]].head()

,review,clean_review,sentiment
0,I would recommend this laptop. It is easy to u...,would recommend laptop easy use great use home...,positive
1,This backpack is not bad at all. It feels exce...,backpack not bad feel excellent worth price us...,positive
2,Poor value for money. The tablet feels bad and...,poor value money tablet feel bad look old sell...,negative
3,Good value for money. The watch feels impressi...,good value money watch feel impressive look st...,positive
4,"After one week, the camera started giving prob...",one week camera start give problem many issue ...,negative


## Step 2: Train/Test Split

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["clean_review"], df["sentiment"], test_size=0.20, random_state=42, stratify=df["sentiment"]
)
len(X_train), len(X_test)

(288, 72)

## Step 3: TF-IDF + Classifier

TF-IDF converts text into numerical arrays. Unigrams and bigrams help capture words and phrases such as `not good`.

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

models = {
    "MultinomialNB": Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1,2), max_features=10000, min_df=2, sublinear_tf=True)), ("model", MultinomialNB(alpha=1.0))]),
    "ComplementNB": Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1,2), max_features=10000, min_df=2, sublinear_tf=True)), ("model", ComplementNB(alpha=1.0))]),
    "LinearSVC": Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1,2), max_features=10000, min_df=2, sublinear_tf=True)), ("model", LinearSVC(C=1.0, random_state=42))]),
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, pred, average="weighted", zero_division=0)
    results.append({"model": name, "accuracy": accuracy_score(y_test, pred), "precision": precision, "recall": recall, "f1_score": f1})

pd.DataFrame(results).sort_values("f1_score", ascending=False)

,model,accuracy,precision,recall,f1_score
2,LinearSVC,1.000000,1.000000,1.000000,1.000000
0,MultinomialNB,0.986111,0.986486,0.986111,0.986108
1,ComplementNB,0.986111,0.986486,0.986111,0.986108


## Step 4: Run Full Training Script

The script saves the best model, metrics and charts.

In [5]:
!python src/train_model.py

Preprocessing text...
Training MultinomialNB...
Training ComplementNB...
Training LinearSVC...

Training complete.
           model  accuracy  precision  recall  f1_score
2      LinearSVC    1.0000     1.0000  1.0000    1.0000
0  MultinomialNB    0.9861     0.9865  0.9861    0.9861
1   ComplementNB    0.9861     0.9865  0.9861    0.9861
Best model saved to: C:\Users\karan\Desktop\internship\Decode_labs\project 4\Project_4_NLP_Sentiment_Analysis\models\best_sentiment_model.pkl


## Step 5: Prediction Demo

In [6]:
!python src/predict.py "This product is not good and not worth the price"

{'review': 'This product is not good and not worth the price', 'clean_text': 'product not good not worth price', 'prediction': 'negative', 'confidence': 0.6693}
